# Understanding the `add_messages` Reducer in LangGraph

In LangGraph, the **State** is a dictionary that flows between nodes.
By default, whenever a node returns a dictionary key (e.g. `{'messages': [...]}`), LangGraph **completely overwrites** the old value of that key with the new value.

### What is `add_messages`?
`add_messages` is a built-in **reducer function**. When you annotate a state key with `Annotated[list, add_messages]`, you tell LangGraph:
> *"Whenever a node or an input provides new messages, do NOT overwrite the existing list — merge them instead!"*

This notebook demonstrates 4 practical scenarios that explain why `add_messages` is essential:
1. **The Overwrite Problem (Without Reducer vs With Reducer)**
2. **Multi-Node Pipelines (Chaining multiple nodes together)**
3. **Multi-Turn Memory with Checkpointer (`MemorySaver` + `add_messages`)**
4. **Smart Message Updates by ID (In-place edit / deduplication)**



### 0. Setup and Environment Initialization

In [7]:
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq

# Load environment variables (GROQ_API_KEY from .env)
load_dotenv(find_dotenv())

# Initialize LLM
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7,
)



---
## Example 1: The Overwrite Problem (Without vs With `add_messages`)

Let's see what happens if we **do NOT** use `add_messages` vs when we **do** use it.



In [8]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

# --- CASE A: WITHOUT REDUCER (Default Overwrite) ---
class StateWithoutReducer(TypedDict):
    messages: list  # Plain list: LangGraph will overwrite on every update!

def bot_no_reducer(state: StateWithoutReducer):
    # Node returns an AIMessage
    ai_reply = llm.invoke(state["messages"])
    return {"messages": [ai_reply]}

builder_no_reducer = StateGraph(StateWithoutReducer)
builder_no_reducer.add_node("bot", bot_no_reducer)
builder_no_reducer.add_edge(START, "bot")
builder_no_reducer.add_edge("bot", END)
graph_no_reducer = builder_no_reducer.compile()

# Invoke with a user question
result_no_reducer = graph_no_reducer.invoke({
    "messages": [HumanMessage(content="What is the capital of France?")]
})

print("=== WITHOUT REDUCER ===")
print(f"Total messages in state: {len(result_no_reducer['messages'])}")
for msg in result_no_reducer['messages']:
    print(f"- {msg.__class__.__name__}: {msg.content}")



=== WITHOUT REDUCER ===
Total messages in state: 1
- AIMessage: The capital of France is **Paris**.


Notice above: **The `HumanMessage` is GONE!**
Because there was no reducer, returning `{"messages": [AIMessage(...)]}` completely replaced `messages` in the state, destroying the user's original question.

Now let's see the same graph **WITH `add_messages`**:



In [9]:
# --- CASE B: WITH add_messages REDUCER ---
class StateWithReducer(TypedDict):
    messages: Annotated[list, add_messages]  # Append / merge new messages!

def bot_with_reducer(state: StateWithReducer):
    ai_reply = llm.invoke(state["messages"])
    return {"messages": [ai_reply]}

builder_with_reducer = StateGraph(StateWithReducer)
builder_with_reducer.add_node("bot", bot_with_reducer)
builder_with_reducer.add_edge(START, "bot")
builder_with_reducer.add_edge("bot", END)
graph_with_reducer = builder_with_reducer.compile()

result_with_reducer = graph_with_reducer.invoke({
    "messages": [HumanMessage(content="What is the capital of France?")]
})

print("=== WITH add_messages ===")
print(f"Total messages in state: {len(result_with_reducer['messages'])}")
for msg in result_with_reducer['messages']:
    print(f"- {msg.__class__.__name__}: {msg.content}")



=== WITH add_messages ===
Total messages in state: 2
- HumanMessage: What is the capital of France?
- AIMessage: The capital of France is **Paris**.


With `add_messages`, both the **`HumanMessage`** and the **`AIMessage`** are preserved in the state.



---
## Example 2: Multi-Node Pipeline (Why Workflows & Agents Require `add_messages`)

In real applications, graphs have multiple nodes (e.g. `DraftBot` -> `ReviewerBot`, or `Agent` -> `ToolExecutor`).
Each node adds new messages to the conversation.

Without `add_messages`, **Node 2 would erase everything Node 1 did!**
With `add_messages`, all nodes append their messages sequentially to build the complete history.



In [10]:
class PipelineState(TypedDict):
    messages: Annotated[list, add_messages]

def drafter_node(state: PipelineState):
    prompt = state["messages"] + [HumanMessage(content="Provide a quick 1-sentence draft answer.")]
    response = llm.invoke(prompt)
    response.content = f"[Draft]: {response.content}"
    return {"messages": [response]}

def reviewer_node(state: PipelineState):
    # Reviewer inspects the previous draft from state['messages']
    prompt = state["messages"] + [HumanMessage(content="Polish the draft above into a professional tone in 1 sentence.")]
    response = llm.invoke(prompt)
    response.content = f"[Final Polish]: {response.content}"
    return {"messages": [response]}

pipeline = StateGraph(PipelineState)
pipeline.add_node("Drafter", drafter_node)
pipeline.add_node("Reviewer", reviewer_node)

pipeline.add_edge(START, "Drafter")
pipeline.add_edge("Drafter", "Reviewer")
pipeline.add_edge("Reviewer", END)

app_pipeline = pipeline.compile()

result = app_pipeline.invoke({
    "messages": [HumanMessage(content="Write a slogan for an eco-friendly coffee shop.")]
})

print(f"Total messages accumulated across pipeline: {len(result['messages'])}\n")
for msg in result["messages"]:
    print(f"{msg.__class__.__name__}:\n{msg.content}\n")



Total messages accumulated across pipeline: 3

HumanMessage:
Write a slogan for an eco-friendly coffee shop.

AIMessage:
[Draft]: Sip sustainably, brew responsibly.

AIMessage:
[Final Polish]: Sip sustainably and brew responsibly with our commitment to eco-conscious coffee.



---
## Example 3: Multi-Turn Memory (`MemorySaver` + `add_messages`)

### Why did your previous question fail?
When you asked:
> `What is my name and what i play?`

The model responded: *"I don't know your name or what you play..."*

This is because **`graph.invoke()` is stateless by default**. Each call starts a completely new graph with an empty state.

To remember across turns, LangGraph needs **two** things working together:
1. **`checkpointer` (`MemorySaver`)**: Persists state across invocations using a `thread_id`.
2. **`add_messages`**: Merges the new turn's question into the saved state instead of replacing it!



In [11]:
from langgraph.checkpoint.memory import MemorySaver

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

def chat_bot(state: ChatState):
    return {"messages": [llm.invoke(state["messages"])]}

chat_graph = StateGraph(ChatState)
chat_graph.add_node("ChatBot", chat_bot)
chat_graph.add_edge(START, "ChatBot")
chat_graph.add_edge("ChatBot", END)

# 1. Compile with MemorySaver
memory = MemorySaver()
chat_app = chat_graph.compile(checkpointer=memory)

# 2. Assign a session thread_id
config = {"configurable": {"thread_id": "user-session-1"}}

# --- TURN 1 ---
print("--- TURN 1 ---")
turn1 = chat_app.invoke(
    {"messages": [HumanMessage(content="Hi, My Name is Arpit and I like cricket")]},
    config=config
)
print("Bot:", turn1["messages"][-1].content)
print("\n" + "="*50 + "\n")

# --- TURN 2 (Now it remembers!) ---
print("--- TURN 2 ---")
turn2 = chat_app.invoke(
    {"messages": [HumanMessage(content="What is my name and what do I play?")]},
    config=config
)
print("Bot:", turn2["messages"][-1].content)



--- TURN 1 ---
Bot: Hi Arpit! Nice to meet you.

Cricket is a fantastic sport with such rich history and strategy. Do you have a favorite team or player that you follow closely? Or are you more interested in the technical aspects, like batting or bowling techniques? I'd love to hear what you enjoy most about it! 🏏


--- TURN 2 ---
Bot: Your name is **Arpit**, and based on our conversation, you like **cricket**.

However, you didn't explicitly state that you *play* cricket—you only mentioned that you *like* it. If you do play, I'd be happy to know more about your skill level or favorite role (batting, bowling, wicket-keeping, etc.)! 😊


---
## Example 4: Smart Message Updates by ID (Deduplication / Editing)

`add_messages` is much smarter than a standard Python `list.append()`.

If you provide a message with an `id` that **already exists** in the state, `add_messages` **updates that message in place** rather than adding a duplicate!
This is used for streaming updates, retrying failed steps, or human-in-the-loop message editing.



In [ ]:
# Direct demonstration of add_messages function
from langchain_core.messages import HumanMessage, AIMessage

# Step 1: Initial conversation
existing_messages = [
    HumanMessage(id="msg-1", content="Tell me a joke"),
    AIMessage(id="msg-2", content="Why did the chicken cross the road?"),
]

# Step 2: New update contains an existing ID ("msg-2") with updated content
updated_messages = [
    AIMessage(id="msg-2", content="Why did the chicken cross the road? To get to the other side! (UPDATED PUNCHLINE)")
]

# Merge using add_messages
merged = add_messages(existing_messages, updated_messages)

print(f"Original message count: {len(existing_messages)}")
print(f"Merged message count:   {len(merged)} (No duplicates added!)\n")

for msg in merged:
    print(f"[{msg.id}] {msg.content}")



---
## Summary Cheat Sheet

| Feature | Without `add_messages` | With `add_messages` |
| :--- | :--- | :--- |
| **Node update behavior** | Overwrites the `messages` list completely | Appends new messages to the list |
| **Preserve user prompt?** | ❌ No, user prompt is erased when bot responds | ✅ Yes, both prompt and response stay |
| **Multi-node workflows** | ❌ Nodes overwrite each other's outputs | ✅ History accumulates across all nodes |
| **Checkpointer persistence** | ❌ New turn wipes out checkpointed state | ✅ New turn appends to saved history |
| **Message with matching ID** | N/A (overwrites all) | Updates the message in-place |

